# FOML Project — Classical ML, PCA and LDA (Baseline)

**Project:** AI-generated vs. real image detection  
**Input:** `split_foml_dataset.zip` created in the previous step  
**Goal:** Build an explainable classical-ML baseline using PCA and LDA, then compare a few standard classifiers.

## Important rules
- This notebook uses **only the training set to fit PCA and classifiers**.
- It uses the **validation set** to compare candidate models.
- It deliberately does **not load or evaluate the test set**. We will reserve the test set until the classical ML and CNN approaches are ready for a fair final comparison.
- The input images are resized to **32 × 32 only for this classical baseline** to keep the flattened pixel feature size manageable. The processed dataset remains unchanged; later our CNN can use 256 × 256 images.
- A very small dataset means validation scores can change noticeably with a few predictions. Treat this as a baseline, not proof of a general-purpose detector.

Run cells from top to bottom. The code favors straightforward Python and comments that can be explained in a viva.


## 1. Upload the split dataset ZIP

In [ ]:
from pathlib import Path
import zipfile

ZIP_PATH = Path('/content/split_foml_dataset.zip')

if not ZIP_PATH.exists():
    from google.colab import files
    uploaded = files.upload()  # Select split_foml_dataset.zip
    zip_files = [name for name in uploaded if name.lower().endswith('.zip')]
    if len(zip_files) != 1:
        raise ValueError('Please upload exactly one ZIP file: split_foml_dataset.zip')
    ZIP_PATH = Path('/content') / zip_files[0]

EXTRACT_DIR = Path('/content/foml_classical_ml_data')
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(EXTRACT_DIR)

# Find the folder containing train/, validation/, and test/.
ROOT = None
for candidate in EXTRACT_DIR.rglob('train'):
    if candidate.is_dir() and (candidate.parent / 'validation').is_dir() and (candidate.parent / 'test').is_dir():
        ROOT = candidate.parent
        break

if ROOT is None:
    raise FileNotFoundError('Could not find train/, validation/, and test/ folders inside the ZIP.')

print('Dataset folder:', ROOT)
print('We will load train and validation images only. The test set stays reserved.')


## 2. Import libraries and list the train/validation images

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
import joblib

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
LABEL_TO_NUMBER = {'real': 0, 'ai_generated': 1}
NUMBER_TO_LABEL = {0: 'Real', 1: 'AI-generated'}


def list_images(split_name):
    """Collect image paths and their labels/categories from one split folder."""
    records = []
    split_folder = ROOT / split_name
    for image_path in sorted(split_folder.rglob('*')):
        if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
            # Folder structure: split/label/category/image.jpg
            label = image_path.parent.parent.name
            category = image_path.parent.name
            if label not in LABEL_TO_NUMBER:
                continue
            records.append({
                'path': image_path,
                'label': label,
                'label_id': LABEL_TO_NUMBER[label],
                'category': category,
                'split': split_name,
            })
    return records

train_records = list_images('train')
val_records = list_images('validation')

counts = pd.DataFrame([
    {'split': 'train', 'label': label, 'count': sum(r['label'] == label for r in train_records)}
    for label in ['real', 'ai_generated']
] + [
    {'split': 'validation', 'label': label, 'count': sum(r['label'] == label for r in val_records)}
    for label in ['real', 'ai_generated']
])
display(counts)

print(f'Training images: {len(train_records)}')
print(f'Validation images: {len(val_records)}')
print('Note: test images have not been loaded or used.')

if len(train_records) == 0 or len(val_records) == 0:
    raise ValueError('Training or validation images were not found. Check the ZIP folder structure.')


## 3. Preview some training images

This is a quick visual check. The folder names provide the labels; no model has been trained yet.


In [ ]:
# Show a few examples from the training split only.
rng = np.random.default_rng(42)
chosen_indices = rng.choice(len(train_records), size=min(8, len(train_records)), replace=False)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, idx in zip(axes.flat, chosen_indices):
    record = train_records[idx]
    with Image.open(record['path']) as img:
        ax.imshow(img.convert('RGB'))
    ax.set_title(f"{record['label']}\n{record['category']}", fontsize=9)
    ax.axis('off')
for ax in list(axes.flat)[len(chosen_indices):]:
    ax.axis('off')
plt.tight_layout()
plt.show()


## 4. Convert images to simple pixel features

For this first classical-ML experiment, each image is resized to **32 × 32 pixels** and represented by its RGB pixel values.

- 32 × 32 × 3 = **3,072 numbers per image**.
- Pixel values are scaled to the range 0–1.
- We use the same conversion for real and AI-generated images.
- We do not change the source ZIP or the processed dataset.

Raw pixels are a simple baseline. They may not capture the most useful forensic clues, so we will interpret the results cautiously.


In [ ]:
CLASSICAL_IMAGE_SIZE = (32, 32)


def image_to_features(image_path):
    """Resize one image, convert it to RGB, then flatten pixel values."""
    with Image.open(image_path) as image:
        image = image.convert('RGB')
        image = image.resize(CLASSICAL_IMAGE_SIZE)
        array = np.asarray(image, dtype=np.float32) / 255.0
    return array.reshape(-1)


def records_to_arrays(records):
    X = np.stack([image_to_features(record['path']) for record in records])
    y = np.array([record['label_id'] for record in records], dtype=np.int64)
    return X, y

X_train, y_train = records_to_arrays(train_records)
X_val, y_val = records_to_arrays(val_records)

print('X_train shape:', X_train.shape)
print('X_val shape:  ', X_val.shape)
print('Each image has', X_train.shape[1], 'pixel features.')
print('Training class counts:', dict(zip(*np.unique(y_train, return_counts=True))))


## 5. PCA: reduce the number of features

PCA (Principal Component Analysis) transforms the original pixel features into a smaller set of combinations called principal components. It is **unsupervised**: it does not use the labels while finding those components.

We fit PCA using the training images only, then apply the already-fitted PCA to validation images. This prevents validation data from influencing the fitted transformation.


In [ ]:
PCA_COMPONENTS = 30

pca = PCA(n_components=PCA_COMPONENTS, random_state=42)
X_train_pca = pca.fit_transform(X_train)  # Fit on training data only
X_val_pca = pca.transform(X_val)         # Transform validation data only

print('Original features per image:', X_train.shape[1])
print('PCA features per image:     ', X_train_pca.shape[1])
print('Variance retained by 30 components:', round(pca.explained_variance_ratio_.sum(), 4))


### Optional visualization: PCA in 2D

The plot below is only a visualization. The models above use 30 PCA components, not two.


In [ ]:
pca_2d = PCA(n_components=2, random_state=42)
train_2d = pca_2d.fit_transform(X_train)
val_2d = pca_2d.transform(X_val)

plt.figure(figsize=(8, 5))
for label_id, label_name, color in [(0, 'Real', 'tab:blue'), (1, 'AI-generated', 'tab:orange')]:
    train_mask = y_train == label_id
    val_mask = y_val == label_id
    plt.scatter(train_2d[train_mask, 0], train_2d[train_mask, 1], label=f'{label_name} (train)', alpha=0.7, marker='o')
    plt.scatter(val_2d[val_mask, 0], val_2d[val_mask, 1], label=f'{label_name} (validation)', alpha=0.9, marker='x')
plt.xlabel('Principal component 1')
plt.ylabel('Principal component 2')
plt.title('PCA visualization of real and AI-generated images')
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()


## 6. LDA: a supervised method

LDA (Linear Discriminant Analysis) uses the class labels to find directions that separate the classes. With two classes, LDA can make at most one discriminant direction.

We will first visualize a one-dimensional LDA projection, then include an LDA classifier in our model comparison.


In [ ]:
lda_projection = LinearDiscriminantAnalysis(n_components=1)
train_lda_1d = lda_projection.fit_transform(X_train_pca, y_train).ravel()

plt.figure(figsize=(8, 4))
plt.hist(train_lda_1d[y_train == 0], bins=12, alpha=0.65, label='Real')
plt.hist(train_lda_1d[y_train == 1], bins=12, alpha=0.65, label='AI-generated')
plt.xlabel('LDA discriminant score')
plt.ylabel('Number of training images')
plt.title('LDA projection of training images')
plt.legend()
plt.tight_layout()
plt.show()


## 7. Train and compare classical ML classifiers

All four models use the same 30 PCA features so the comparison is easier to understand:

1. **Logistic Regression** — a simple linear classifier.
2. **LDA classifier** — a linear classifier that models class separation.
3. **Support Vector Machine (SVM)** — finds a separating boundary; we use an RBF kernel to allow a non-linear boundary.
4. **Random Forest** — combines predictions from many decision trees.

The validation scores are used to compare the candidates. We will not choose settings based on the test set.


In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42),
    'LDA classifier': LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto'),
    'SVM (RBF)': SVC(kernel='rbf', C=1.0, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=200, class_weight='balanced_subsample', random_state=42, n_jobs=-1),
}

results = []
validation_predictions = {}

for model_name, model in models.items():
    model.fit(X_train_pca, y_train)
    predictions = model.predict(X_val_pca)
    validation_predictions[model_name] = predictions

    results.append({
        'model': model_name,
        'accuracy': accuracy_score(y_val, predictions),
        'precision_ai': precision_score(y_val, predictions, pos_label=1, zero_division=0),
        'recall_ai': recall_score(y_val, predictions, pos_label=1, zero_division=0),
        'f1_ai': f1_score(y_val, predictions, pos_label=1, zero_division=0),
    })

results_df = pd.DataFrame(results).sort_values('f1_ai', ascending=False).reset_index(drop=True)
display(results_df.round(3))
print('Reminder: validation has only 30 images, so each prediction has a noticeable effect on the scores.')


## 8. Confusion matrices on the validation set

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, (model_name, predictions) in zip(axes.flat, validation_predictions.items()):
    cm = confusion_matrix(y_val, predictions, labels=[0, 1])
    display_obj = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Real', 'AI-generated'])
    display_obj.plot(ax=ax, colorbar=False, values_format='d')
    ax.set_title(model_name)
plt.tight_layout()
plt.show()


## 9. Save results and candidate models

This cell saves:
- `classical_ml_validation_results.csv` — validation metrics for the candidate models.
- `classical_ml_models.joblib` — the fitted PCA transformer and candidate classifiers.

Keep these files for later. We will compare the classical approaches with our CNN before deciding which model to evaluate on the reserved test set. The test set is intentionally not evaluated in this notebook.


In [ ]:
RESULTS_PATH = Path('/content/classical_ml_validation_results.csv')
MODEL_PATH = Path('/content/classical_ml_models.joblib')

results_df.to_csv(RESULTS_PATH, index=False)
model_bundle = {
    'pca': pca,
    'models': models,
    'image_size': CLASSICAL_IMAGE_SIZE,
    'label_to_number': LABEL_TO_NUMBER,
    'number_to_label': NUMBER_TO_LABEL,
    'note': 'Models trained on train split; compared on validation split. Test split intentionally not evaluated.'
}
joblib.dump(model_bundle, MODEL_PATH)

print('Saved:', RESULTS_PATH)
print('Saved:', MODEL_PATH)

try:
    from google.colab import files
    files.download(str(RESULTS_PATH))
    files.download(str(MODEL_PATH))
except Exception:
    print('Not running in Colab; files are saved at the paths above.')


## 10. How to interpret the first results

- Do not worry if the first accuracy is modest. This is a small dataset and raw pixels are a basic feature representation.
- If training accuracy is much higher than validation accuracy, the model may be overfitting.
- Do not keep changing settings based on the test set. It stays reserved for final evaluation.
- Do not add more data automatically based on one score. We will review the metrics, confusion matrices, and errors first.
- Next, we will build a small CNN, then compare classical ML and CNN results using the same fixed split.
